# Document Question Answering System (RAG)

A Retrieval-Augmented Generation (RAG) pipeline that answers questions grounded in your own custom documents (PDFs, notes, resumes, research papers, books) instead of relying only on a language model's internal (and possibly outdated or hallucinated) knowledge.

**Pipeline stages:**
1. **Document Ingestion** — load PDFs / text files into raw text
2. **Text Chunking** — split into overlapping chunks for better retrieval granularity
3. **Embedding Creation** — convert each chunk into a semantic vector (open-source `sentence-transformers` model)
4. **Vector Database** — store embeddings in a FAISS index for fast similarity search
5. **Query Processing** — embed the user's question the same way
6. **Context Retrieval** — retrieve the top-k most relevant chunks
7. **Answer Generation** — an open-source language model (`google/flan-t5-base`) generates an answer grounded in the retrieved context

This notebook uses **only open-source / free models** (no API keys, no Anthropic/OpenAI calls needed) — everything runs locally or on a free Colab GPU/CPU.


## 1. Install dependencies

- `pypdf` — read PDF files
- `sentence-transformers` — open-source embedding model
- `faiss-cpu` — vector similarity search / vector database
- `transformers` + `torch` — open-source generation model (FLAN-T5)
- `datasets` — optional, to load the `vectara/open_ragbench` dataset from Hugging Face


In [11]:
!pip install -q pypdf python-docx sentence-transformers faiss-cpu transformers torch datasets accelerate

## 2. Imports

In [12]:
import os
import glob
import numpy as np
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline

import warnings
warnings.filterwarnings("ignore")


## 3. Document Ingestion

Loads text out of `.pdf`, `.txt`, and `.md` files. Point `DOCS_PATH` at a folder containing your own documents (notes, resume, research papers, books, etc.).

If you don't have your own documents handy yet, this notebook creates a small sample text file so you can try the pipeline immediately.


In [13]:
DOCS_PATH = "./sample_docs"
os.makedirs(DOCS_PATH, exist_ok=True)

# Create a sample document if the folder is empty, so the notebook is runnable out of the box.
if not os.listdir(DOCS_PATH):
    sample_text = '''Acme Robotics Employee Handbook

1. Working Hours
Standard working hours are 9:00 AM to 5:30 PM, Monday through Friday.
Employees may request flexible hours by speaking with their manager.
Core collaboration hours, during which all employees should be reachable,
are 11:00 AM to 3:00 PM.

2. Remote Work Policy
Employees may work remotely up to 3 days per week. Requests for full-time
remote work must be approved by the department head and HR. All remote
employees must be reachable via Slack during core hours.

3. Leave Policy
Full-time employees accrue 18 days of paid time off (PTO) per year, plus
10 public holidays. Sick leave is separate and capped at 12 days per year.
Unused PTO can be carried over up to a maximum of 5 days into the next
calendar year.

4. Expense Reimbursement
Employees can be reimbursed for business-related expenses such as travel,
client meals, and approved software subscriptions. All expense reports
must be submitted within 30 days of the expense and include an itemized
receipt. Reimbursements are processed within 10 business days.

5. Performance Reviews
Performance reviews are conducted twice a year, in June and December.
Reviews include a self-assessment, manager feedback, and peer feedback
from at least two colleagues. Promotions are considered during the
December review cycle.

6. Equipment Policy
New employees receive a laptop, monitor, and a one-time $300 stipend for
home office equipment. Equipment must be returned within 14 days of
departure from the company.
'''
    with open(os.path.join(DOCS_PATH, "company_handbook.txt"), "w") as f:
        f.write(sample_text)

print("Documents found:", os.listdir(DOCS_PATH))


Documents found: ['company_handbook.txt']


In [14]:
def load_txt(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def load_pdf(filepath):
    reader = PdfReader(filepath)
    pages_text = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages_text.append(f"\n[Page {i + 1}]\n{text}")
    return "\n".join(pages_text)


def load_docx(filepath):
    import docx
    doc = docx.Document(filepath)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())


def load_document(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext in (".txt", ".md"):
        return load_txt(filepath)
    elif ext == ".pdf":
        return load_pdf(filepath)
    elif ext == ".docx":
        return load_docx(filepath)
    else:
        raise ValueError(f"Unsupported file type: {ext}")


def load_folder(folder_path):
    supported = {".txt", ".md", ".pdf", ".docx"}
    docs = {}
    for fname in sorted(os.listdir(folder_path)):
        ext = os.path.splitext(fname)[1].lower()
        if ext in supported:
            full_path = os.path.join(folder_path, fname)
            try:
                docs[fname] = load_document(full_path)
            except Exception as e:
                print(f"  [!] Skipping {fname}: {e}")
    return docs


documents = load_folder(DOCS_PATH)
print(f"Loaded {len(documents)} document(s): {list(documents.keys())}")


Loaded 1 document(s): ['company_handbook.txt']


## 4. Text Chunking

Long documents are split into smaller overlapping chunks. Overlap preserves context across chunk boundaries so a sentence's meaning isn't cut in half.


In [15]:
def chunk_text(text, source="document", chunk_size=800, overlap=150):
    text = text.strip()
    if not text:
        return []

    chunks = []
    start = 0
    chunk_id = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)

        if end < text_len:
            next_space = text.find(" ", end)
            if next_space != -1 and next_space - end < 50:
                end = next_space

        piece = text[start:end].strip()
        if piece:
            chunks.append({"text": piece, "source": source, "chunk_id": chunk_id})
            chunk_id += 1

        if end >= text_len:
            break
        start = end - overlap if end - overlap > start else end

    return chunks


def chunk_documents(docs, chunk_size=800, overlap=150):
    all_chunks = []
    for source, text in docs.items():
        all_chunks.extend(chunk_text(text, source=source, chunk_size=chunk_size, overlap=overlap))
    return all_chunks


chunks = chunk_documents(documents, chunk_size=800, overlap=150)
print(f"Split {len(documents)} document(s) into {len(chunks)} chunks")
print("\nExample chunk:\n", chunks[0]["text"][:300], "...")


Split 1 document(s) into 3 chunks

Example chunk:
 Acme Robotics Employee Handbook

1. Working Hours
Standard working hours are 9:00 AM to 5:30 PM, Monday through Friday.
Employees may request flexible hours by speaking with their manager.
Core collaboration hours, during which all employees should be reachable,
are 11:00 AM to 3:00 PM.

2. Remote W ...


## 5. Embedding Creation

Each chunk is converted into a dense vector using an open-source `sentence-transformers` model (`all-MiniLM-L6-v2`) — small, fast, and runs on CPU.


In [16]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # normalize so we can use inner-product == cosine similarity
)

print("Embedding matrix shape:", chunk_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (3, 384)


## 6. Vector Database (FAISS)

Store the chunk embeddings in a FAISS index for fast approximate/exact nearest-neighbor search. Since embeddings are normalized, inner product search is equivalent to cosine similarity.


In [17]:
embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)   # inner product = cosine similarity (vectors are normalized)
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors of dimension {embedding_dim}")


FAISS index built with 3 vectors of dimension 384


## 7. Query Processing + Context Retrieval

Embed the user's question with the *same* embedding model, then search the FAISS index for the most similar chunks.


In [18]:
def retrieve(query, top_k=4):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append((chunks[idx], float(score)))
    return results


# Quick sanity check
test_results = retrieve("How many PTO days do employees get?", top_k=3)
for chunk, score in test_results:
    print(f"[{chunk['source']} #{chunk['chunk_id']}] score={score:.3f}")
    print(chunk['text'][:150].replace(chr(10), ' '), "...\n")


[company_handbook.txt #0] score=0.630
Acme Robotics Employee Handbook  1. Working Hours Standard working hours are 9:00 AM to 5:30 PM, Monday through Friday. Employees may request flexible ...

[company_handbook.txt #1] score=0.626
ped at 12 days per year. Unused PTO can be carried over up to a maximum of 5 days into the next calendar year.  4. Expense Reimbursement Employees can ...

[company_handbook.txt #2] score=0.277
mber review cycle.  6. Equipment Policy New employees receive a laptop, monitor, and a one-time $300 stipend for home office equipment. Equipment must ...



## 8. Answer Generation (open-source LLM)

Instead of a paid API, this uses **FLAN-T5** (`google/flan-t5-base`), a free, open-source instruction-tuned model from Google, run locally via Hugging Face `transformers`. It's small enough to run on CPU.

You can swap in a larger/different open-source model here (e.g. `google/flan-t5-large`, `MBZUAI/LaMini-Flan-T5-783M`, or a local `Llama`/`Mistral` model) without changing the rest of the pipeline.


In [19]:
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
)

def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(f"[{c['source']}]: {c['text']}" for c, _ in retrieved_chunks)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer isn't in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    return prompt


def generate_answer(question, retrieved_chunks):
    prompt = build_prompt(question, retrieved_chunks)
    output = generator(prompt, max_new_tokens=256, do_sample=False)
    return output[0]["generated_text"]


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

## 9. Full RAG Pipeline

Combine retrieval + generation into a single function representing the whole flow:

**User Question → Retrieve relevant chunks → Provide as context → Generate grounded answer**


In [20]:
def rag_answer(question, top_k=4, show_context=False):
    retrieved = retrieve(question, top_k=top_k)

    if show_context:
        print("--- Retrieved context ---")
        for chunk, score in retrieved:
            print(f"[{chunk['source']} #{chunk['chunk_id']}] score={score:.3f}")
            print(chunk['text'][:200].replace(chr(10), ' '), "...\n")

    answer = generate_answer(question, retrieved)
    return answer


## 10. Try it out

Ask questions about the documents in `./sample_docs` (or point `DOCS_PATH` above at your own PDFs/notes/resume/research papers and re-run cells 3–6).


In [21]:
question = "How many PTO days do employees get and can they carry it over?"
answer = rag_answer(question, top_k=3, show_context=True)

print("=== Answer ===")
print(answer)


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


--- Retrieved context ---
[company_handbook.txt #1] score=0.641
ped at 12 days per year. Unused PTO can be carried over up to a maximum of 5 days into the next calendar year.  4. Expense Reimbursement Employees can be reimbursed for business-related expenses such  ...

[company_handbook.txt #0] score=0.613
Acme Robotics Employee Handbook  1. Working Hours Standard working hours are 9:00 AM to 5:30 PM, Monday through Friday. Employees may request flexible hours by speaking with their manager. Core collab ...

[company_handbook.txt #2] score=0.312
mber review cycle.  6. Equipment Policy New employees receive a laptop, monitor, and a one-time $300 stipend for home office equipment. Equipment must be returned within 14 days of departure from the  ...



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Answer ===
Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
[company_handbook.txt]: ped at 12 days per year.
Unused PTO can be carried over up to a maximum of 5 days into the next
calendar year.

4. Expense Reimbursement
Employees can be reimbursed for business-related expenses such as travel,
client meals, and approved software subscriptions. All expense reports
must be submitted within 30 days of the expense and include an itemized
receipt. Reimbursements are processed within 10 business days.

5. Performance Reviews
Performance reviews are conducted twice a year, in June and December.
Reviews include a self-assessment, manager feedback, and peer feedback
from at least two colleagues. Promotions are considered during the
December review cycle.

6. Equipment Policy
New employees receive a laptop, monitor, and a one-time $300 stipend for
home office equipment. Equipment must

[company_handbook.txt]: Acme Robotics Em

In [22]:
# Try your own question here
question = "What is the remote work policy?"
answer = rag_answer(question, top_k=3)
print("=== Answer ===")
print(answer)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Answer ===
Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
[company_handbook.txt]: Acme Robotics Employee Handbook

1. Working Hours
Standard working hours are 9:00 AM to 5:30 PM, Monday through Friday.
Employees may request flexible hours by speaking with their manager.
Core collaboration hours, during which all employees should be reachable,
are 11:00 AM to 3:00 PM.

2. Remote Work Policy
Employees may work remotely up to 3 days per week. Requests for full-time
remote work must be approved by the department head and HR. All remote
employees must be reachable via Slack during core hours.

3. Leave Policy
Full-time employees accrue 18 days of paid time off (PTO) per year, plus
10 public holidays. Sick leave is separate and capped at 12 days per year.
Unused PTO can be carried over up to a maximum of 5 days into the next
calendar year.

4. Expense Reimbursement
Employees can

[company_handbook.txt]: ped at 12 days p

## 11. (Optional) Using the `open_ragbench` dataset instead of your own documents

If you'd rather test on a ready-made benchmark dataset instead of your own PDFs, you can load the [`vectara/open_ragbench`](https://huggingface.co/datasets/vectara/open_ragbench) dataset from Hugging Face and feed its documents through the same pipeline (steps 3–6 above use `documents`, a `{filename: text}` dict, so just populate it from the dataset).


In [23]:
from datasets import load_dataset

# Uncomment to try it (requires internet access to huggingface.co):
# ragbench = load_dataset("vectara/open_ragbench", split="train[:20]")  # small slice for a quick test
# documents = {f"doc_{i}": row["text"] for i, row in enumerate(ragbench) if "text" in row}
# chunks = chunk_documents(documents)
# # ... then re-run the embedding (step 5) and FAISS index (step 6) cells with the new `chunks`


## 12. Improvements & Experiments

- **Better chunking** — try sentence-aware or semantic chunking instead of fixed character windows
- **Different embedding models** — e.g. `all-mpnet-base-v2` (higher quality, slower) or domain-specific embedding models
- **Hybrid search** — combine keyword search (BM25) with vector search for better recall
- **Re-ranking** — use a cross-encoder (e.g. `cross-encoder/ms-marco-MiniLM-L-6-v2`) to re-rank the top retrieved chunks before generation
- **Different generation models** — swap FLAN-T5 for a larger open-source LLM (Llama, Mistral, Gemma) for more fluent answers
- **Persistent vector store** — swap the in-memory FAISS index for `Chroma` or a persisted FAISS index so you don't need to re-embed documents every session

## Key Learnings

- How RAG combines **retrieval** (finding relevant information) with **generation** (producing a natural-language answer), which improves factual grounding over relying on a language model's parametric knowledge alone
- The importance of good chunking and embedding choices for retrieval quality
- How vector databases (FAISS) enable fast similarity search over large document collections
- How to design a modular, swappable AI pipeline (any component — embedding model, vector store, or generator — can be replaced independently)

## Conclusion

This notebook builds a working Retrieval-Augmented Generation system entirely with open-source components — no paid API required. RAG systems like this are the foundation of chatbots, knowledge assistants, enterprise search tools, and AI-powered documentation systems that need to answer questions grounded in private or domain-specific data.
